# Lib


In [16]:
import warnings

warnings.filterwarnings("ignore")

In [17]:
import os
import copy
from pathlib import Path

import cv2
import numpy as np
import pandas as pd

import albumentations as A
from albumentations.pytorch import ToTensorV2

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision.models import (
    resnet34,
    resnet50,
    ResNet34_Weights,
    ResNet50_Weights,
)

from scipy.ndimage import binary_erosion, distance_transform_edt

import kagglehub

In [18]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cpu


# Unet architecture


In [19]:
DEFAULT_EVAL_CONFIG = {
    "device": "cuda" if torch.cuda.is_available() else "cpu",
    "image_height": 224,
    "image_width": 224,
    "num_classes": 3,
    # U-Net architecture: must match training checkpoint.
    "backbone": "resnet50",
    "bilinear": False,
    # Evaluation data.
    "batch_size": 16,
    # Windows/Jupyter-safe; Linux users may increase this if needed.
    "num_workers": 0,
    "image_col": None,
    "mask_col": None,
    "dataset_handle": "avcuongy/bkai-igh-neopolyp-small-sample",
    # Input preprocessing.
    "normalize_input": True,
    "input_mean": [0.485, 0.456, 0.406],
    "input_std": [0.229, 0.224, 0.225],
    # Checkpoint loading.
    "strict_weight_loading": True,
}

BASE_CLASS_NAMES = {
    0: "background",
    1: "non-neoplastic",
    2: "neoplastic",
}

In [20]:
def decode_ground_truth(mask_rgb):
    mask = np.asarray(mask_rgb)

    if mask.ndim == 2:
        unique_values = np.unique(mask)

        if np.isin(unique_values, [0, 1, 2]).all():
            return mask.astype(np.uint8)

        raise ValueError(
            "Grayscale mask must contain only class IDs 0, 1, 2. "
            f"Found: {unique_values[:20]}"
        )

    if mask.ndim != 3 or mask.shape[-1] != 3:
        raise ValueError(f"Expected HxWx3 RGB mask, got shape={mask.shape}")

    colors = np.array(
        [
            [0, 0, 0],
            [0, 255, 0],
            [255, 0, 0],
        ],
        dtype=np.float32,
    )

    distances = np.sum(
        (mask.astype(np.float32)[:, :, None, :] - colors[None, None, :, :]) ** 2,
        axis=-1,
    )

    return np.argmin(distances, axis=-1).astype(np.uint8)


def pick_column(
    dataframe,
    explicit_column=None,
    candidates=None,
    required=True,
):
    if explicit_column is not None:
        if explicit_column not in dataframe.columns:
            raise KeyError(
                f"Column '{explicit_column}' not found. "
                f"Available: {dataframe.columns.tolist()}"
            )
        return explicit_column

    normalized = {str(column).strip().lower(): column for column in dataframe.columns}

    for candidate in candidates or []:
        if candidate.lower() in normalized:
            return normalized[candidate.lower()]

    if len(dataframe.columns) == 1:
        return dataframe.columns[0]

    if required:
        raise KeyError(
            "Cannot infer CSV column automatically. "
            f"Available: {dataframe.columns.tolist()}"
        )

    return None


class SegmentationEvaluationDataset(Dataset):
    IMAGE_COLUMN_CANDIDATES = [
        "image",
        "image_name",
        "image_path",
        "filename",
        "file_name",
        "file",
        "id",
    ]

    MASK_COLUMN_CANDIDATES = [
        "mask",
        "mask_name",
        "mask_path",
        "label",
        "label_path",
        "gt",
        "gt_path",
    ]

    def __init__(
        self,
        dataframe,
        image_dir,
        mask_dir,
        transform=None,
        image_col=None,
        mask_col=None,
    ):
        self.df = dataframe.reset_index(drop=True).copy()
        self.image_dir = Path(image_dir)
        self.mask_dir = Path(mask_dir)
        self.transform = transform

        self.image_col = pick_column(
            self.df,
            explicit_column=image_col,
            candidates=self.IMAGE_COLUMN_CANDIDATES,
            required=True,
        )

        self.mask_col = pick_column(
            self.df,
            explicit_column=mask_col,
            candidates=self.MASK_COLUMN_CANDIDATES,
            required=False,
        )

    @staticmethod
    def resolve_path(root_dir, value):
        path = Path(str(value))
        return path if path.is_absolute() else Path(root_dir) / path

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        row = self.df.iloc[index]

        image_path = self.resolve_path(
            self.image_dir,
            row[self.image_col],
        )

        mask_value = (
            row[self.mask_col] if self.mask_col is not None else row[self.image_col]
        )

        mask_path = self.resolve_path(
            self.mask_dir,
            mask_value,
        )

        image_bgr = cv2.imread(
            str(image_path),
            cv2.IMREAD_COLOR,
        )
        mask_raw = cv2.imread(
            str(mask_path),
            cv2.IMREAD_UNCHANGED,
        )

        if image_bgr is None:
            raise FileNotFoundError(f"Cannot read image: {image_path}")

        if mask_raw is None:
            raise FileNotFoundError(f"Cannot read mask: {mask_path}")

        image = cv2.cvtColor(
            image_bgr,
            cv2.COLOR_BGR2RGB,
        )

        if mask_raw.ndim == 3:
            mask_raw = cv2.cvtColor(
                mask_raw,
                cv2.COLOR_BGR2RGB,
            )

        mask = decode_ground_truth(mask_raw)

        if self.transform is not None:
            transformed = self.transform(
                image=image,
                mask=mask,
            )
            image = transformed["image"]
            mask = transformed["mask"]

        if not torch.is_tensor(mask):
            mask = torch.from_numpy(mask)

        return {
            "image": image,
            "mask": mask.long(),
            "image_path": str(image_path),
            "mask_path": str(mask_path),
            "image_name": image_path.name,
        }


def build_eval_transform(config):
    transform_steps = [
        A.Resize(
            height=int(config["image_height"]),
            width=int(config["image_width"]),
        )
    ]

    if config["normalize_input"]:
        transform_steps.append(
            A.Normalize(
                mean=tuple(config["input_mean"]),
                std=tuple(config["input_std"]),
            )
        )

    transform_steps.append(ToTensorV2())

    return A.Compose(transform_steps)


def build_test_data(config):
    device = torch.device(config["device"])
    eval_transform = build_eval_transform(config)

    dataset_root = Path(kagglehub.dataset_download(config["dataset_handle"]))

    data_dir = dataset_root / "data"
    test_csv_path = data_dir / "sample" / "test.csv"

    if not test_csv_path.exists():
        raise FileNotFoundError(f"Test CSV not found: {test_csv_path}")

    test_df = pd.read_csv(test_csv_path)

    test_dataset = SegmentationEvaluationDataset(
        dataframe=test_df,
        image_dir=data_dir,
        mask_dir=data_dir,
        transform=eval_transform,
        image_col=config["image_col"],
        mask_col=config["mask_col"],
    )

    requested_num_workers = int(config.get("num_workers", 0))

    if requested_num_workers < 0:
        raise ValueError("CONFIG['num_workers'] must be >= 0.")

    effective_num_workers = requested_num_workers

    # Windows starts DataLoader workers with spawn. Dataset classes defined
    # inside a Jupyter notebook are not reliably importable in those workers,
    # which can surface only as "DataLoader worker exited unexpectedly".
    if os.name == "nt" and requested_num_workers > 0:
        print(
            "[DataLoader] Windows notebook detected: "
            f"num_workers={requested_num_workers} -> 0."
        )
        effective_num_workers = 0

    test_loader = DataLoader(
        test_dataset,
        batch_size=int(config["batch_size"]),
        shuffle=False,
        num_workers=effective_num_workers,
        pin_memory=device.type == "cuda",
    )

    return test_df, test_dataset, test_loader

In [21]:
class ResNetEncoder(nn.Module):
    def __init__(
        self,
        backbone="resnet50",
        pretrained=False,
    ):
        super().__init__()

        backbone = backbone.lower()

        if backbone == "resnet34":
            weights = ResNet34_Weights.DEFAULT if pretrained else None
            model = resnet34(weights=weights)
            self.out_channels = [
                64,
                64,
                128,
                256,
                512,
            ]

        elif backbone == "resnet50":
            weights = ResNet50_Weights.DEFAULT if pretrained else None
            model = resnet50(weights=weights)
            self.out_channels = [
                64,
                256,
                512,
                1024,
                2048,
            ]

        else:
            raise ValueError(
                f"Unsupported backbone: {backbone}. " "Supported: resnet34, resnet50."
            )

        self.conv1 = model.conv1
        self.bn1 = model.bn1
        self.relu = model.relu
        self.maxpool = model.maxpool

        self.layer1 = model.layer1
        self.layer2 = model.layer2
        self.layer3 = model.layer3
        self.layer4 = model.layer4

    def forward(self, x):
        x0 = self.relu(self.bn1(self.conv1(x)))

        x = self.maxpool(x0)
        x1 = self.layer1(x)
        x2 = self.layer2(x1)
        x3 = self.layer3(x2)
        x4 = self.layer4(x3)

        return [
            x0,
            x1,
            x2,
            x3,
            x4,
        ]


class DoubleConv(nn.Module):
    def __init__(
        self,
        in_channels,
        out_channels,
    ):
        super().__init__()

        self.double_conv = nn.Sequential(
            nn.Conv2d(
                in_channels,
                out_channels,
                kernel_size=3,
                padding=1,
                bias=False,
            ),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(
                out_channels,
                out_channels,
                kernel_size=3,
                padding=1,
                bias=False,
            ),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.double_conv(x)


class Up(nn.Module):
    def __init__(
        self,
        decoder_channels,
        skip_channels,
        out_channels,
        bilinear=False,
    ):
        super().__init__()

        if bilinear:
            self.up = nn.Sequential(
                nn.Upsample(
                    scale_factor=2,
                    mode="bilinear",
                    align_corners=True,
                ),
                nn.Conv2d(
                    decoder_channels,
                    out_channels,
                    kernel_size=1,
                ),
            )
        else:
            self.up = nn.ConvTranspose2d(
                decoder_channels,
                out_channels,
                kernel_size=2,
                stride=2,
            )

        self.conv = DoubleConv(
            out_channels + skip_channels,
            out_channels,
        )

    def forward(self, x, skip):
        x = self.up(x)

        diff_y = skip.size(2) - x.size(2)
        diff_x = skip.size(3) - x.size(3)

        if diff_x != 0 or diff_y != 0:
            x = F.pad(
                x,
                [
                    diff_x // 2,
                    diff_x - diff_x // 2,
                    diff_y // 2,
                    diff_y - diff_y // 2,
                ],
            )

        x = torch.cat(
            [skip, x],
            dim=1,
        )

        return self.conv(x)


class OutConv(nn.Module):
    def __init__(
        self,
        in_channels,
        n_classes,
    ):
        super().__init__()

        self.conv = nn.Conv2d(
            in_channels,
            n_classes,
            kernel_size=1,
        )

    def forward(self, x):
        return self.conv(x)


class UNet(nn.Module):
    def __init__(
        self,
        backbone="resnet50",
        n_classes=3,
        pretrained=False,
        bilinear=False,
    ):
        super().__init__()

        self.backbone_name = backbone
        self.n_classes = n_classes
        self.bilinear = bilinear

        self.encoder = ResNetEncoder(
            backbone=backbone,
            pretrained=pretrained,
        )

        c0, c1, c2, c3, c4 = self.encoder.out_channels

        self.up4 = Up(
            decoder_channels=c4,
            skip_channels=c3,
            out_channels=256,
            bilinear=bilinear,
        )

        self.up3 = Up(
            decoder_channels=256,
            skip_channels=c2,
            out_channels=128,
            bilinear=bilinear,
        )

        self.up2 = Up(
            decoder_channels=128,
            skip_channels=c1,
            out_channels=64,
            bilinear=bilinear,
        )

        self.up1 = Up(
            decoder_channels=64,
            skip_channels=c0,
            out_channels=64,
            bilinear=bilinear,
        )

        if bilinear:
            self.final_up = nn.Sequential(
                nn.Upsample(
                    scale_factor=2,
                    mode="bilinear",
                    align_corners=True,
                ),
                nn.Conv2d(
                    64,
                    32,
                    kernel_size=1,
                ),
                nn.BatchNorm2d(32),
                nn.ReLU(inplace=True),
            )
        else:
            self.final_up = nn.Sequential(
                nn.ConvTranspose2d(
                    64,
                    32,
                    kernel_size=2,
                    stride=2,
                ),
                nn.BatchNorm2d(32),
                nn.ReLU(inplace=True),
            )

        self.outc = OutConv(
            32,
            n_classes,
        )

    def forward(self, x):
        input_size = x.shape[-2:]

        x0, x1, x2, x3, x4 = self.encoder(x)

        x = self.up4(x4, x3)
        x = self.up3(x, x2)
        x = self.up2(x, x1)
        x = self.up1(x, x0)

        x = self.final_up(x)
        logits = self.outc(x)

        if logits.shape[-2:] != input_size:
            logits = F.interpolate(
                logits,
                size=input_size,
                mode="bilinear",
                align_corners=False,
            )

        return logits

In [22]:
def load_checkpoint_file(
    checkpoint_path,
    map_location="cpu",
):
    checkpoint_path = Path(checkpoint_path)

    if not checkpoint_path.exists():
        raise FileNotFoundError(f"Model checkpoint not found: {checkpoint_path}")

    checkpoint = torch.load(
        checkpoint_path,
        map_location=map_location,
        weights_only=False,
    )

    if not isinstance(checkpoint, dict):
        raise TypeError("Checkpoint must be a dictionary or raw state_dict.")

    return checkpoint


def is_raw_state_dict(value):
    return (
        isinstance(value, dict)
        and bool(value)
        and all(torch.is_tensor(item) for item in value.values())
    )


def extract_model_state_dict(checkpoint):
    if is_raw_state_dict(checkpoint):
        return checkpoint

    for key in (
        "model_state_dict",
        "state_dict",
        "model",
        "net",
        "network",
        "weights",
    ):
        candidate = checkpoint.get(key)

        if isinstance(candidate, nn.Module):
            return candidate.state_dict()

        if is_raw_state_dict(candidate):
            return candidate

        if isinstance(candidate, dict):
            for nested_key in (
                "state_dict",
                "model_state_dict",
                "weights",
            ):
                nested = candidate.get(nested_key)

                if is_raw_state_dict(nested):
                    return nested

    raise KeyError("Cannot find model weights in the checkpoint.")


def strip_prefix(state_dict, prefix):
    return {
        (key[len(prefix) :] if key.startswith(prefix) else key): value
        for key, value in state_dict.items()
    }


def align_state_dict_keys(
    state_dict,
    model,
):
    model_state = model.state_dict()
    candidates = [state_dict]
    candidate = state_dict

    for _ in range(8):
        stripped = False

        for prefix in (
            "module.",
            "model.",
            "net.",
            "network.",
            "_orig_mod.",
        ):
            if candidate and all(key.startswith(prefix) for key in candidate):
                candidate = strip_prefix(
                    candidate,
                    prefix,
                )
                candidates.append(candidate)
                stripped = True
                break

        if not stripped:
            break

    def match_score(candidate_state):
        return sum(
            key in model_state and model_state[key].shape == value.shape
            for key, value in candidate_state.items()
        )

    return max(
        candidates,
        key=match_score,
    )


def validate_checkpoint_config(
    checkpoint,
    model_label,
    config,
):
    saved_config = checkpoint.get("config")

    if not isinstance(saved_config, dict):
        return

    for key in (
        "image_height",
        "image_width",
        "num_classes",
    ):
        saved_value = saved_config.get(key)

        if saved_value is None:
            continue

        if int(saved_value) != int(config[key]):
            raise ValueError(
                f"{model_label}: checkpoint {key}="
                f"{saved_value}, but evaluation "
                f"{key}={config[key]}."
            )

    saved_backbone = saved_config.get("backbone")

    if (
        saved_backbone is not None
        and str(saved_backbone).lower() != str(config["backbone"]).lower()
    ):
        raise ValueError(
            f"{model_label}: checkpoint backbone="
            f"{saved_backbone}, but evaluation "
            f"backbone={config['backbone']}."
        )


def load_weights_into_model(
    model,
    checkpoint,
    model_label,
    strict=True,
):
    state_dict = align_state_dict_keys(
        extract_model_state_dict(checkpoint),
        model,
    )

    result = model.load_state_dict(
        state_dict,
        strict=bool(strict),
    )

    print(
        f"[{model_label}] "
        f"missing={len(result.missing_keys)} | "
        f"unexpected={len(result.unexpected_keys)}"
    )

    return result


def build_and_load_unet(
    checkpoint_path,
    model_label,
    config,
):
    model = UNet(
        backbone=config["backbone"],
        n_classes=int(config["num_classes"]),
        pretrained=False,
        bilinear=bool(config["bilinear"]),
    )

    checkpoint = load_checkpoint_file(
        checkpoint_path,
        map_location="cpu",
    )

    validate_checkpoint_config(
        checkpoint=checkpoint,
        model_label=model_label,
        config=config,
    )

    load_weights_into_model(
        model=model,
        checkpoint=checkpoint,
        model_label=model_label,
        strict=config["strict_weight_loading"],
    )

    model.eval()

    return model

In [23]:
def extract_unet_logits(
    model_output,
    model_label,
):
    if torch.is_tensor(model_output):
        logits = model_output

    elif isinstance(model_output, (tuple, list)):
        tensors = [item for item in model_output if torch.is_tensor(item)]
        logits = next(
            (item for item in tensors if item.ndim == 4),
            None,
        )

    elif isinstance(model_output, dict):
        logits = None

        for key in (
            "logits",
            "out",
            "prediction",
            "pred",
        ):
            value = model_output.get(key)

            if torch.is_tensor(value):
                logits = value
                break

    elif hasattr(model_output, "logits") and torch.is_tensor(model_output.logits):
        logits = model_output.logits

    else:
        logits = None

    if logits is None:
        raise TypeError(f"{model_label}: cannot extract logits " "from model output.")

    if logits.ndim != 4:
        raise ValueError(
            f"{model_label}: expected logits "
            f"[B, C, H, W], got {tuple(logits.shape)}."
        )

    return logits


@torch.inference_mode()
def predict_unet_batch(
    model,
    images,
    model_label,
    num_classes,
    output_size=None,
):
    logits = extract_unet_logits(
        model(images),
        model_label,
    )

    if logits.shape[1] != num_classes:
        raise ValueError(
            f"{model_label}: expected "
            f"{num_classes} output channels, "
            f"got {logits.shape[1]}."
        )

    if output_size is not None and logits.shape[-2:] != tuple(output_size):
        logits = F.interpolate(
            logits,
            size=tuple(output_size),
            mode="bilinear",
            align_corners=False,
        )

    return torch.argmax(
        logits,
        dim=1,
    )


def hd95_binary(
    pred_mask,
    true_mask,
):
    pred = np.asarray(
        pred_mask,
        dtype=bool,
    )
    true = np.asarray(
        true_mask,
        dtype=bool,
    )

    pred_empty = not pred.any()
    true_empty = not true.any()

    # Correct absence on both sides.
    if pred_empty and true_empty:
        return 0.0

    # Complete miss or complete false-positive case.
    if pred_empty or true_empty:
        height, width = true.shape
        return float(
            np.hypot(
                height,
                width,
            )
        )

    pred_surface = pred ^ binary_erosion(pred)
    true_surface = true ^ binary_erosion(true)

    if not pred_surface.any():
        pred_surface = pred

    if not true_surface.any():
        true_surface = true

    distance_to_true = distance_transform_edt(~true_surface)
    distance_to_pred = distance_transform_edt(~pred_surface)

    distances = np.concatenate(
        [
            distance_to_true[pred_surface],
            distance_to_pred[true_surface],
        ]
    )

    return float(
        np.percentile(
            distances,
            95,
        )
    )


def safe_divide(
    numerator,
    denominator,
):
    if denominator == 0:
        return np.nan

    return float(numerator / denominator)


@torch.inference_mode()
def evaluate_unet_model(
    model,
    model_label,
    loader,
    device,
    num_classes,
    class_names,
):
    confusion = np.zeros(
        (num_classes, num_classes),
        dtype=np.int64,
    )

    hd95_values = {class_id: [] for class_id in range(num_classes)}

    model = model.to(device)
    model.eval()

    for batch in loader:
        images = batch["image"].to(
            device,
            non_blocking=True,
        )
        targets = (
            batch["mask"]
            .to(
                device,
                non_blocking=True,
            )
            .long()
        )

        predictions = predict_unet_batch(
            model=model,
            images=images,
            model_label=model_label,
            num_classes=num_classes,
            output_size=targets.shape[-2:],
        )

        targets_np = targets.cpu().numpy()
        predictions_np = predictions.cpu().numpy()

        encoded = num_classes * targets_np.reshape(-1) + predictions_np.reshape(-1)

        confusion += np.bincount(
            encoded,
            minlength=num_classes**2,
        ).reshape(
            num_classes,
            num_classes,
        )

        for sample_index in range(targets_np.shape[0]):
            for class_id in range(num_classes):
                hd95_values[class_id].append(
                    hd95_binary(
                        predictions_np[sample_index] == class_id,
                        targets_np[sample_index] == class_id,
                    )
                )

    rows = []

    for class_id in range(num_classes):
        true_positive = int(
            confusion[
                class_id,
                class_id,
            ]
        )

        false_positive = int(confusion[:, class_id].sum() - true_positive)

        false_negative = int(confusion[class_id, :].sum() - true_positive)

        gt_pixels = int(confusion[class_id, :].sum())

        pred_pixels = int(
            confusion[
                :,
                class_id,
            ].sum()
        )

        hd_values = np.asarray(
            hd95_values[class_id],
            dtype=np.float64,
        )

        rows.append(
            {
                "model": model_label,
                "class_id": class_id,
                "class_name": class_names[class_id],
                "dice": safe_divide(
                    2 * true_positive,
                    (2 * true_positive + false_positive + false_negative),
                ),
                "iou": safe_divide(
                    true_positive,
                    (true_positive + false_positive + false_negative),
                ),
                "hd95": (float(np.mean(hd_values)) if hd_values.size > 0 else np.nan),
                "precision": safe_divide(
                    true_positive,
                    (true_positive + false_positive),
                ),
                "recall": safe_divide(
                    true_positive,
                    (true_positive + false_negative),
                ),
                "gt_pixels": gt_pixels,
                "pred_pixels": pred_pixels,
            }
        )

    class_results = pd.DataFrame(rows)

    foreground = class_results[class_results["class_id"] > 0]

    metric_columns = [
        "dice",
        "iou",
        "hd95",
        "precision",
        "recall",
    ]

    def macro_row(
        class_id,
        class_name,
        frame,
    ):
        row = {
            "model": model_label,
            "class_id": class_id,
            "class_name": class_name,
            "gt_pixels": int(frame["gt_pixels"].sum()),
            "pred_pixels": int(frame["pred_pixels"].sum()),
        }

        row.update(
            {column: frame[column].mean(skipna=True) for column in metric_columns}
        )

        return row

    results = pd.concat(
        [
            class_results,
            pd.DataFrame(
                [
                    macro_row(
                        "macro_fg",
                        "macro_foreground_only",
                        foreground,
                    ),
                    macro_row(
                        "macro_all",
                        "macro_all_classes",
                        class_results,
                    ),
                ]
            ),
        ],
        ignore_index=True,
    )

    model.to("cpu")

    if device.type == "cuda":
        torch.cuda.empty_cache()

    return results, confusion


def evaluate_unet_checkpoints(
    models,
    loader,
    device,
    num_classes,
    class_names,
):
    evaluation_tables = {}
    confusion_matrices = {}

    for model_label, model in models.items():
        print(f"Evaluating: {model_label}")

        model_results, model_confusion = evaluate_unet_model(
            model=model,
            model_label=model_label,
            loader=loader,
            device=device,
            num_classes=num_classes,
            class_names=class_names,
        )

        evaluation_tables[model_label] = model_results

        confusion_matrices[model_label] = model_confusion

    evaluation_df = pd.concat(
        evaluation_tables.values(),
        ignore_index=True,
    )

    model_comparison_df = evaluation_df[
        evaluation_df["class_id"].eq("macro_fg")
    ].reset_index(drop=True)

    return (
        evaluation_df,
        model_comparison_df,
        confusion_matrices,
    )

# Main


In [24]:
CONFIG = copy.deepcopy(DEFAULT_EVAL_CONFIG)

device = torch.device(CONFIG["device"])

class_names = {
    class_id: BASE_CLASS_NAMES.get(
        class_id,
        f"class_{class_id}",
    )
    for class_id in range(int(CONFIG["num_classes"]))
}

test_df, test_dataset, test_loader = build_test_data(CONFIG)

print("Device      :", device)
print("Test samples:", len(test_dataset))
print("DataLoader workers:", test_loader.num_workers)

Device      : cpu
Test samples: 200
DataLoader workers: 0


In [ ]:
CHECKPOINTS = {
    "unet_resnet_half": r"..\models\baseline\best_UNet_half.pth",
    "unet_resnet_quarter": r"..\models\baseline\best_UNet_quarter.pth",
}

In [26]:
UNET_MODELS = {
    model_label: build_and_load_unet(
        checkpoint_path=checkpoint_path,
        model_label=model_label,
        config=CONFIG,
    )
    for model_label, checkpoint_path in CHECKPOINTS.items()
}

print(
    "Loaded models:",
    list(UNET_MODELS),
)

[unet_resnet_half] missing=0 | unexpected=0
[unet_resnet50_quarter] missing=0 | unexpected=0
Loaded models: ['unet_resnet_half', 'unet_resnet50_quarter']


In [27]:
(
    evaluation_df,
    model_comparison_df,
    confusion_matrices,
) = evaluate_unet_checkpoints(
    models=UNET_MODELS,
    loader=test_loader,
    device=device,
    num_classes=int(CONFIG["num_classes"]),
    class_names=class_names,
)

Evaluating: unet_resnet_half
Evaluating: unet_resnet50_quarter


In [28]:
evaluation_df.round(4)

,model,class_id,class_name,dice,iou,hd95,precision,recall,gt_pixels,pred_pixels
0,unet_resnet_half,0,background,0.9926,0.9854,5.8985,0.9927,0.9926,9469136,9467869
1,unet_resnet_half,1,non-neoplastic,0.4985,0.3320,84.8771,0.4769,0.5222,56416,61783
2,unet_resnet_half,2,neoplastic,0.8585,0.7520,70.7573,0.8619,0.8550,509648,505548
3,unet_resnet_half,macro_fg,macro_foreground_only,0.6785,0.5420,77.8172,0.6694,0.6886,566064,567331
4,unet_resnet_half,macro_all,macro_all_classes,0.7832,0.6898,53.8443,0.7772,0.7899,10035200,10035200
5,unet_resnet50_quarter,0,background,0.9917,0.9834,7.2712,0.9925,0.9908,9469136,9453141
6,unet_resnet50_quarter,1,non-neoplastic,0.4447,0.2859,123.6352,0.3578,0.5874,56416,92611
7,unet_resnet50_quarter,2,neoplastic,0.8407,0.7253,70.9887,0.8581,0.8241,509648,489448
8,unet_resnet50_quarter,macro_fg,macro_foreground_only,0.6427,0.5056,97.3120,0.6080,0.7057,566064,582059
9,unet_resnet50_quarter,macro_all,macro_all_classes,0.7590,0.6649,67.2984,0.7361,0.8008,10035200,10035200


In [29]:
model_comparison_df.round(4)

,model,class_id,class_name,dice,iou,hd95,precision,recall,gt_pixels,pred_pixels
0,unet_resnet_half,macro_fg,macro_foreground_only,0.6785,0.5420,77.8172,0.6694,0.6886,566064,567331
1,unet_resnet50_quarter,macro_fg,macro_foreground_only,0.6427,0.5056,97.3120,0.6080,0.7057,566064,582059


# Inference Visualization

Hai kiểu visualize theo `unet_core.ipynb`:

1. Random `n_sample` ảnh từ `test_dataset`, mỗi ảnh một hàng:
   **Original | Ground Truth | Prediction | Overlay**
2. Inference theo `IMAGE_PATH` chỉ định; `GROUND_TRUTH_PATH` là tùy chọn.

In [ ]:
VIS_CLASS_COLORS = {
    0: (0, 0, 0),
    1: (0, 255, 0),
    2: (255, 0, 0),
}


def class_mask_to_rgb(class_mask, class_colors=VIS_CLASS_COLORS):
    """Convert class-index mask -> RGB mask."""
    class_mask = np.asarray(class_mask)

    rgb = np.zeros(
        (*class_mask.shape, 3),
        dtype=np.uint8,
    )

    for class_id, color in class_colors.items():
        rgb[class_mask == class_id] = color

    return rgb


def _read_rgb_image(image_path):
    image_path = Path(image_path)

    if not image_path.exists():
        raise FileNotFoundError(f"Image not found: {image_path}")

    image_bgr = cv2.imread(
        str(image_path),
        cv2.IMREAD_COLOR,
    )

    if image_bgr is None:
        raise FileNotFoundError(f"Cannot read image: {image_path}")

    return cv2.cvtColor(
        image_bgr,
        cv2.COLOR_BGR2RGB,
    )


def _read_gt_class_mask(ground_truth_path):
    ground_truth_path = Path(ground_truth_path)

    if not ground_truth_path.exists():
        raise FileNotFoundError(
            f"Ground truth not found: {ground_truth_path}"
        )

    gt_raw = cv2.imread(
        str(ground_truth_path),
        cv2.IMREAD_UNCHANGED,
    )

    if gt_raw is None:
        raise FileNotFoundError(
            f"Cannot read ground truth: {ground_truth_path}"
        )

    if gt_raw.ndim == 3:
        gt_raw = cv2.cvtColor(
            gt_raw,
            cv2.COLOR_BGR2RGB,
        )

    return decode_ground_truth(gt_raw)


@torch.inference_mode()
def inference_unet_image(
    model,
    model_label,
    image_path,
    transform,
    device,
    num_classes,
    restore_original_size=True,
):
    """
    Inference one image with the same U-Net prediction path used by evaluation.

    Returns:
        pred_mask: HxW class-index mask.
    """
    model = model.to(device)
    model.eval()

    image_rgb = _read_rgb_image(image_path)
    original_height, original_width = image_rgb.shape[:2]

    transformed = transform(image=image_rgb)
    image_tensor = transformed["image"].unsqueeze(0).to(
        device,
        non_blocking=True,
    )

    pred_mask = predict_unet_batch(
        model=model,
        images=image_tensor,
        model_label=model_label,
        num_classes=int(num_classes),
        output_size=None,
    )[0].cpu().numpy().astype(np.uint8)

    if restore_original_size:
        pred_mask = cv2.resize(
            pred_mask,
            (original_width, original_height),
            interpolation=cv2.INTER_NEAREST,
        )

    return pred_mask


def visualize_unet_inference_samples(
    model,
    model_label,
    dataset,
    device,
    num_classes,
    n_sample=4,
    alpha=0.40,
    seed=None,
):
    """
    Random n_sample from test dataset.

    Each image is displayed on ONE ROW:
        Original | Ground Truth | Prediction | Overlay
    """
    if n_sample <= 0:
        raise ValueError("n_sample must be greater than 0.")

    if len(dataset) == 0:
        raise ValueError("Dataset is empty.")

    if dataset.transform is None:
        raise ValueError("Dataset transform is required for inference.")

    n_sample = min(int(n_sample), len(dataset))

    if seed is None:
        sample_indices = np.random.choice(
            len(dataset),
            size=n_sample,
            replace=False,
        )
    else:
        rng = np.random.default_rng(int(seed))
        sample_indices = rng.choice(
            len(dataset),
            size=n_sample,
            replace=False,
        )

    fig, axes = plt.subplots(
        n_sample,
        4,
        figsize=(20, 5 * n_sample),
    )

    if n_sample == 1:
        axes = np.expand_dims(axes, axis=0)

    for row, index in enumerate(sample_indices):
        # Use the paths already resolved by the evaluation dataset.
        sample = dataset[int(index)]
        image_path = Path(sample["image_path"])
        gt_path = Path(sample["mask_path"])

        pred_mask = inference_unet_image(
            model=model,
            model_label=model_label,
            image_path=image_path,
            transform=dataset.transform,
            device=device,
            num_classes=num_classes,
            restore_original_size=True,
        )

        image_rgb = _read_rgb_image(image_path)
        gt_class = _read_gt_class_mask(gt_path)

        gt_rgb = class_mask_to_rgb(gt_class)
        pred_rgb = class_mask_to_rgb(pred_mask)

        if gt_rgb.shape[:2] != image_rgb.shape[:2]:
            gt_rgb = cv2.resize(
                gt_rgb,
                (image_rgb.shape[1], image_rgb.shape[0]),
                interpolation=cv2.INTER_NEAREST,
            )

        if pred_rgb.shape[:2] != image_rgb.shape[:2]:
            pred_rgb = cv2.resize(
                pred_rgb,
                (image_rgb.shape[1], image_rgb.shape[0]),
                interpolation=cv2.INTER_NEAREST,
            )

        overlay = cv2.addWeighted(
            image_rgb,
            1.0 - float(alpha),
            pred_rgb,
            float(alpha),
            0,
        )

        panels = [
            (image_rgb, f"Original\n{image_path.name}"),
            (gt_rgb, "Ground Truth"),
            (pred_rgb, "Prediction"),
            (overlay, "Overlay"),
        ]

        for col, (panel, title) in enumerate(panels):
            axes[row, col].imshow(panel)
            axes[row, col].set_title(title)
            axes[row, col].axis("off")

    fig.suptitle(
        f"Random test inference — {model_label}",
        fontsize=14,
        y=1.002,
    )
    plt.tight_layout(pad=0.5)
    plt.show()


def visualize_unet_inference_path(
    model,
    model_label,
    image_path,
    transform,
    device,
    num_classes,
    ground_truth_path=None,
    alpha=0.40,
):
    """
    Inference and visualize one explicitly specified image path.

    With ground_truth_path:
        Original | Ground Truth | Prediction | Overlay

    Without ground_truth_path:
        Original | Prediction | Overlay
    """
    image_path = Path(image_path)

    pred_mask = inference_unet_image(
        model=model,
        model_label=model_label,
        image_path=image_path,
        transform=transform,
        device=device,
        num_classes=num_classes,
        restore_original_size=True,
    )

    image_rgb = _read_rgb_image(image_path)
    pred_rgb = class_mask_to_rgb(pred_mask)

    if pred_rgb.shape[:2] != image_rgb.shape[:2]:
        pred_rgb = cv2.resize(
            pred_rgb,
            (image_rgb.shape[1], image_rgb.shape[0]),
            interpolation=cv2.INTER_NEAREST,
        )

    overlay = cv2.addWeighted(
        image_rgb,
        1.0 - float(alpha),
        pred_rgb,
        float(alpha),
        0,
    )

    if ground_truth_path is None:
        _, axes = plt.subplots(
            1,
            3,
            figsize=(16, 5),
        )

        panels = [
            (image_rgb, f"Original\n{image_path.name}"),
            (pred_rgb, "Prediction"),
            (overlay, "Overlay"),
        ]
    else:
        gt_class = _read_gt_class_mask(ground_truth_path)
        gt_rgb = class_mask_to_rgb(gt_class)

        if gt_rgb.shape[:2] != image_rgb.shape[:2]:
            gt_rgb = cv2.resize(
                gt_rgb,
                (image_rgb.shape[1], image_rgb.shape[0]),
                interpolation=cv2.INTER_NEAREST,
            )

        _, axes = plt.subplots(
            1,
            4,
            figsize=(20, 5),
        )

        panels = [
            (image_rgb, f"Original\n{image_path.name}"),
            (gt_rgb, "Ground Truth"),
            (pred_rgb, "Prediction"),
            (overlay, "Overlay"),
        ]

    for axis, (panel, title) in zip(axes, panels):
        axis.imshow(panel)
        axis.set_title(title)
        axis.axis("off")

    plt.tight_layout()
    plt.show()

    return pred_mask

## 1. Random `n_sample` from test dataset

In [ ]:
# Select which loaded U-Net checkpoint to visualize.
VIS_MODEL_LABEL = next(iter(UNET_MODELS))
VIS_MODEL = UNET_MODELS[VIS_MODEL_LABEL]

N_SAMPLE = 4
VIS_SEED = None  # e.g. 42 for repeatable random samples

visualize_unet_inference_samples(
    model=VIS_MODEL,
    model_label=VIS_MODEL_LABEL,
    dataset=test_dataset,
    device=device,
    num_classes=int(CONFIG["num_classes"]),
    n_sample=N_SAMPLE,
    alpha=0.40,
    seed=VIS_SEED,
)

## 2. Inference by specified image path

In [ ]:
# Set a specific image path here.
IMAGE_PATH = None
GROUND_TRUTH_PATH = None  # optional

if IMAGE_PATH is None:
    print(
        "Set IMAGE_PATH to run path-based inference. "
        "Set GROUND_TRUTH_PATH too if you want the GT panel."
    )
else:
    visualize_unet_inference_path(
        model=VIS_MODEL,
        model_label=VIS_MODEL_LABEL,
        image_path=IMAGE_PATH,
        transform=test_dataset.transform,
        device=device,
        num_classes=int(CONFIG["num_classes"]),
        ground_truth_path=GROUND_TRUTH_PATH,
        alpha=0.40,
    )